<a href="https://colab.research.google.com/github/lygitdata/GarmentIQ/blob/gh-pages/ipynb/web_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GarmentIQ Segmentation API (Colab + ngrok)

1. Runtime → Change runtime type → **GPU** (recommended)
2. Run all cells in order
3. When prompted, paste your ngrok authtoken (get it at https://dashboard.ngrok.com/get-started/your-authtoken)

In [ ]:
# Install dependencies
!pip install -q flask flask-cors pyngrok garmentiq

In [ ]:
import os
import base64
import shutil
import threading
from getpass import getpass

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok, conf
import garmentiq as giq
from garmentiq.segmentation.model_definition.birefnet import BiRefNet, load_birefnet_config

In [ ]:
# Download BiRefNet model
!mkdir -p ./models/birefnet

!wget -q -O ./models/birefnet/model.safetensors \
    https://huggingface.co/lygitdata/BiRefNet_garmentiq_backup/resolve/main/model.safetensors

In [ ]:
app = Flask(__name__)
CORS(app)

# where we put uploads & outputs
BASE_DIR   = 'images'
INPUT_DIR  = os.path.join(BASE_DIR, 'input')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(INPUT_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# load model once
print("Loading BiRefNet model…")
birefnet = giq.segmentation.load_model(
    model_class=BiRefNet,
    model_path="./models/birefnet/model.safetensors",
    model_args=load_birefnet_config()
)
print("Model loaded.")

In [ ]:
@app.route('/health')
def health_check():
    return jsonify({
        "status": "active",
        "token": "fe9def684914b305540615b3bba461fbf8ea58f460c72b1fe128d3cef93fe4f8",
        "model": "web_segmentation",
        "version": "1.0.0"
    })

@app.route('/segment', methods=['POST'])
def segment():
    files = request.files.getlist('images')
    if not files:
        return jsonify(error='No images uploaded'), 400

    # Ensure input/output directories are fresh
    for d in (INPUT_DIR, OUTPUT_DIR):
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)

    # Save uploaded images
    for f in files:
        f.save(os.path.join(INPUT_DIR, os.path.basename(f.filename)))

    # Parse optional RGB values
    r = request.form.get('red', '')
    g = request.form.get('green', '')
    b = request.form.get('blue', '')
    background_color = None
    if r != '' and g != '' and b != '':
        try:
            rgb = [int(r), int(g), int(b)]
            if any(c < 0 or c > 255 for c in rgb):
                raise ValueError
            background_color = rgb
        except ValueError:
            return jsonify(error='Invalid RGB values; must be integers 0–255'), 400

    # Prepare processing arguments
    kwargs = {
        'image_dir': INPUT_DIR,
        'output_dir': OUTPUT_DIR,
        'model': birefnet,
        'resize_dim': (1024, 1024),
        'normalize_mean': [0.485, 0.456, 0.406],
        'normalize_std': [0.229, 0.224, 0.225]
    }
    if background_color:
        kwargs['background_color'] = background_color

    # Run segmentation
    giq.segmentation.process_and_save_images(**kwargs)

    # Collect masks
    masks = []
    masks_folder = os.path.join(OUTPUT_DIR, 'masks')
    if os.path.isdir(masks_folder):
        for fn in sorted(os.listdir(masks_folder)):
            path = os.path.join(masks_folder, fn)
            if os.path.isfile(path):
                with open(path, 'rb') as imgf:
                    b64 = base64.b64encode(imgf.read()).decode('utf-8')
                masks.append({'filename': fn, 'base64': b64})

    # Collect background-modified images if any
    modified = []
    if background_color:
        mod_folder = os.path.join(OUTPUT_DIR, 'bg_modified')
        if os.path.isdir(mod_folder):
            for fn in sorted(os.listdir(mod_folder)):
                path = os.path.join(mod_folder, fn)
                if os.path.isfile(path):
                    with open(path, 'rb') as imgf:
                        b64 = base64.b64encode(imgf.read()).decode('utf-8')
                    modified.append({'filename': fn, 'base64': b64})

    # Return results
    return jsonify(masks=masks, bg_modified=modified)

# API URL and NGROK token

In [ ]:
# --- Expose the API with ngrok ---
PORT = 5000

# Ask for the token at runtime (input is hidden, nothing is stored in the notebook)
ngrok_token = getpass("Enter your ngrok authtoken: ").strip()
if not ngrok_token:
    raise ValueError("An ngrok authtoken is required.")
conf.get_default().auth_token = ngrok_token
del ngrok_token

# Close any tunnels left over from a previous run
ngrok.kill()

# Start Flask in a background thread so the cell returns and the tunnel stays up
server_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False),
    daemon=True,
)
server_thread.start()

# Open the public tunnel
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url
print("\n✅ API is live!")
print(f"Public URL : {public_url}")

# Kill the API

In [ ]:
# Run this cell to shut everything down (tunnel closes; stop the Colab runtime to end the Flask thread)
ngrok.kill()
print("ngrok tunnel closed.")